# Exercice 2 — Mini-GPT caractère par caractère

Ce notebook construit un petit Transformer autorégressif depuis zéro. Chaque position prédit le caractère suivant. Le masque causal interdit de consulter les caractères futurs.

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

texte = (
    "Les transformeurs traitent des séquences grâce à l'attention."
    "Un modèle causal lit le contexte gauche et prédit le caractère suivant."
    "L'apprentissage ajuste les paramètres pour produire un texte plausible."
    * 80
)
caracteres = sorted(set(texte))
stoi = {c: i for i, c in enumerate(caracteres)}
itos = {i: c for c, i in stoi.items()}
tokens = torch.tensor([stoi[c] for c in texte], dtype=torch.long)
taille_vocabulaire = len(caracteres)
longueur = 64
separation = int(0.9 * len(tokens))
train_tokens, val_tokens = tokens[:separation], tokens[separation:]

def batch(source, taille_batch=32):
    indices = torch.randint(0, len(source) - longueur - 1, (taille_batch,))
    x = torch.stack([source[i:i + longueur] for i in indices])
    y = torch.stack([source[i + 1:i + longueur + 1] for i in indices])
    return x.to(device), y.to(device)

print("Vocabulaire :", taille_vocabulaire, "caractères | device :", device)

Vocabulaire : 31 caractères | device : cpu


## 1. Bloc GPT

Chaque bloc applique une normalisation avant l'attention, une attention multi-têtes causale, une connexion résiduelle, puis un MLP `64 → 256 → 64` avec GELU et une seconde connexion résiduelle.

In [3]:
class BlocGPT(nn.Module):
    def __init__(self, dimension=64, tetes=4):
        super().__init__()
        self.norm_attention = nn.LayerNorm(dimension)
        self.attention = nn.MultiheadAttention(
            dimension, tetes, batch_first=True, dropout=0.0
        )
        self.norm_mlp = nn.LayerNorm(dimension)
        self.mlp = nn.Sequential(
            nn.Linear(64, 256),
            nn.GELU(),
            nn.Linear(256, 64),
        )

    def forward(self, x):
        t = x.shape[1]
        masque = torch.triu(
            torch.ones(t, t, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        attention = self.norm_attention(x)
        attention, _ = self.attention(
            attention, attention, attention, attn_mask=masque, need_weights=False
        )
        x = x + attention
        return x + self.mlp(self.norm_mlp(x))


class MiniGPT(nn.Module):
    def __init__(self, vocabulaire, longueur_max=64):
        super().__init__()
        self.tokens = nn.Embedding(vocabulaire, 64)
        self.positions = nn.Embedding(longueur_max, 64)
        self.blocs = nn.Sequential(BlocGPT(), BlocGPT())
        self.norm_finale = nn.LayerNorm(64)
        self.projection = nn.Linear(64, vocabulaire)

    def forward(self, x, cibles=None):
        _, t = x.shape
        positions = torch.arange(t, device=x.device)
        z = self.tokens(x) + self.positions(positions)[None, :, :]
        logits = self.projection(self.norm_finale(self.blocs(z)))
        perte = None
        if cibles is not None:
            perte = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), cibles.reshape(-1))
        return logits, perte


modele = MiniGPT(taille_vocabulaire).to(device)
print("Logits :", tuple(modele(batch(train_tokens, 2)[0])[0].shape))

Logits : (2, 64, 31)


## 2. Entraînement

Les cibles sont décalées d'un caractère : `x[t]` doit prédire `y[t] = x[t+1]`. La perte d'entropie croisée est calculée sur toutes les positions.

In [4]:
optimiseur = torch.optim.AdamW(modele.parameters(), lr=3e-4)
historique = []
for etape in range(500):
    modele.train()
    x, y = batch(train_tokens)
    _, perte = modele(x, y)
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()
    if (etape + 1) % 100 == 0:
        modele.eval()
        with torch.no_grad():
            _, perte_val = modele(*batch(val_tokens))
        historique.append((perte.item(), perte_val.item()))
        print(f"Étape {etape + 1:03d} | train : {perte.item():.4f} | validation : {perte_val.item():.4f}")

Étape 100 | train : 2.2217 | validation : 2.1932
Étape 200 | train : 1.7186 | validation : 1.7409
Étape 300 | train : 1.2582 | validation : 1.2423
Étape 400 | train : 0.7778 | validation : 0.7562
Étape 500 | train : 0.4627 | validation : 0.3982


## 3. Vérifier le masque causal

En mode évaluation, modifier un caractère futur ne doit pas modifier les logits des positions précédentes.

In [5]:
modele.eval()
entree, _ = batch(val_tokens, 1)
modifiee = entree.clone()
position_future = 20
modifiee[0, position_future] = (modifiee[0, position_future] + 1) % taille_vocabulaire
with torch.no_grad():
    logits_originals, _ = modele(entree)
    logits_modifies, _ = modele(modifiee)
difference = (logits_originals[0, :position_future] - logits_modifies[0, :position_future]).abs().max().item()
print(f"Différence avant la position future : {difference:.8f}")
assert difference < 1e-5

Différence avant la position future : 0.00000000


## 4. Générer 200 caractères

La température est appliquée avant le tirage aléatoire : une valeur basse rend le texte plus prévisible, une valeur élevée augmente la diversité.

In [6]:
def genere(amorce, temperature=1.0, nombre=200):
    modele.eval()
    ids = torch.tensor([[stoi[c] for c in amorce]], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(nombre):
            contexte = ids[:, -longueur:]
            logits, _ = modele(contexte)
            probabilites = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            prochain = torch.multinomial(probabilites, num_samples=1)
            ids = torch.cat([ids, prochain], dim=1)
    return "".join(itos[i] for i in ids[0].tolist())


amorce = "Les transformeurs"
for temperature in (0.5, 1.0, 1.5):
    print(f"\n--- température {temperature} ---")
    print(genere(amorce, temperature=temperature, nombre=200))


--- température 0.5 ---
Les transformeurs traitent des séquent s cegrâce l'attrntention mèle modèle gre gat causatte saucaucauconte litet let le prauce susntivaus ppprarmères lentibles pr podibles pres pres praunsirameusforaite.Leurantes tes

--- température 1.0 ---
Les transformeurs tra deurait s deplaprentisis pcteurage podira te pon lauramète te presitre plairodire pran us sformètesire texte.L'an teurenraiteus sibles gratète séqupâce grâce lstttte pre pames préques poraibraueu

--- température 1.5 ---
Les transformeurs tra deuuivae.Lejursfontrs ceuitent des ues lenjuext ple p'al ahentlentgréL'arbcesas tten ttimàxtes odenmèjue.he tit.varn.pprore civaplenjust xt.varesausss lersfoqurscentesuibl ueurs prmeueuuititcent 


## Questions — réponses

- **Sans masque causal**, une position pourrait voir les caractères futurs pendant l'entraînement. La tâche deviendrait une fuite d'information et la génération réelle serait moins fiable.
- **La température** inférieure à 1 concentre les probabilités ; supérieure à 1 les aplatit et produit des tirages plus variés mais souvent moins cohérents.
- **Le décalage des cibles** aligne chaque entrée sur le caractère qu'elle doit prédire : le caractère présent fournit le contexte du caractère suivant.